# 🌿 01e — Pipeline pollen & moisissures (RNSA)

Construit `dim_pollen.parquet` (dept × annee_mois). 
Lancer `00_config_commun.ipynb` avant.

Source : RNSA (Réseau National de Surveillance Aérobiologique).
https://www.data.gouv.fr/datasets/donnees-historiques-de-surveillance-des-pollens-et-des-moisissures
Fichiers dans `data/raw/BDD_daily_2020_2025/` : 133 capteurs entre 1987 et
2024, un fichier par capteur et par année, données par jour et par taxon.

**Couverture temporelle : 2020-2024 uniquement, il manque 2025.** Le RNSA
a été placé en liquidation judiciaire le 26 mars 2025 et n'a jamais publié
de données pour 2025 (aucun fichier `2025` disponible côté source). Atom France a repris la surveillance pollinique.

**Limite :** les fichiers RNSA n'ont aucune coordonnée
lat/lon, juste un nom de ville, donc pas de reverse geocoding possible comme
pour MENSQ/AASQA. Certaines villes du dataset n'ont pas de correspondance départementale fiable — à vérifier.

In [1]:
# Préambule : on se place dans le répertoire racine du projet et on ajoute le répertoire courant au PYTHONPATH pour pouvoir importer src/config.py

# pour recharger automatiquement les modules modifiés （src config surtout） sans redémarrer le kernel
%load_ext autoreload 
%autoreload 2

import os
import sys
from pathlib import Path
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src.config import RAW_DIR, TABLES_DIR, ANNEE_DEBUT, ANNEE_FIN, DEPTS,  DEPT_NOM_TO_CODE
from src.validation import valider_dim_table

print(
    f"Config chargée depuis src/config.py : {len(DEPTS)} départements | {ANNEE_DEBUT}–{ANNEE_FIN}")
print(f"RAW_DIR    = {RAW_DIR}")
print(f"TABLES_DIR = {TABLES_DIR}")

Config chargée depuis src/config.py : 96 départements | 2020–2025
RAW_DIR    = /Users/siranh/Documents/Data Scientest/projet_liora/data/raw
TABLES_DIR = /Users/siranh/Documents/Data Scientest/projet_liora/data/processed


In [2]:
import re

pollen_dir = Path("data/raw/BDD_daily_2020_2025")
villes = set()
for f in pollen_dir.glob("*.xls"):
    match = re.search(r'Particle_Extract_(.+?)_\d{4}', f.name)
    if match:
        villes.add(match.group(1))

print(f"Total villes : {len(villes)}")
for v in sorted(villes):
    print(f"  {v}")

Total villes : 93
  AGEN
  AIXENPRO
  AJACCIO
  AMBERIEU
  AMIENS
  ANDORRA
  ANGERS
  ANGOULEM
  ANNECY
  ANNEMASS
  ANTONY
  AURILLAC
  AVIGNON
  BAGNOLS
  BART
  BERRIAS
  BESANCON
  BLETTERA
  BORDPESS
  BOURGENB
  BOURGES
  BOURGOIN
  BREST
  BRUSSLAN
  CAEN
  CASTRES
  CHALON-S
  CHAMBERY
  CHARLEVI
  CHAUMONT
  CHOLET
  CLERMONT
  DIJON
  DINAN
  DOLE
  DRAGUIGN
  GAP
  GENAS
  GLEIZE
  GONESSE
  GRENOBLE
  LAROCHE-
  LAROCHL
  LEMANS
  LEPUYENV
  LILLE
  LIMOGES
  LORIENT
  LURE
  LYON
  MACON
  MAREUIL
  MARSEILL
  METZ
  MONTLUCO
  MONTPELL
  MTMARSAN
  MULHOUSE
  NANCY
  NANTES
  NARBONNE
  NEVERS
  NICE
  NICE2
  NIORT
  ORLEANS
  PARIS
  PERIGUEU
  POITIERS
  PONTIVY
  REIMS
  RENNES
  ROANNE
  ROUEN
  ROUSSILL
  SACLAY
  SACLAYSP
  SAINT-ET
  SAINTDIE
  ST-BRIEU
  STALBAN
  STEFOY
  STRASBOU
  TOULON
  TOULOUSE
  TOULOUSM
  TOURS
  TROYES
  TULLE
  VALDAHON
  VALENCE
  VICHY
  VILLENEU


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# TABLE DE CORRESPONDANCE : ville RNSA → code département
# ══════════════════════════════════════════════════════════════════════════════
# Construite à partir des 93 villes du dataset

# TODO : difficile de rattacher les villes au département...

VILLE_TO_DEPT = {
    "AGEN":       "47",  # Lot-et-Garonne
    "AIXENPRO":   "13",  # Bouches-du-Rhône (Aix-en-Provence)
    "AJACCIO":    "2A",  # Corse-du-Sud
    "AMBERIEU":   "01",  # Ain (Ambérieu-en-Bugey)
    "AMIENS":     "80",  # Somme
    "ANDORRA":    None,  # Andorre → hors France, ignoré
    "ANGERS":     "49",  # Maine-et-Loire
    "ANGOULEM":   "16",  # Charente (Angoulême)
    "ANNECY":     "74",  # Haute-Savoie
    "ANNEMASS":   "74",  # Haute-Savoie (Annemasse)
    "ANTONY":     "92",  # Hauts-de-Seine
    "AURILLAC":   "15",  # Cantal
    "AVIGNON":    "84",  # Vaucluse
    "BAGNOLS":    "30",  # Gard (Bagnols-sur-Cèze)
    "BART":       "25",  # Doubs
    "BERRIAS":    "07",  # Ardèche
    "BESANCON":   "25",  # Doubs
    "BEZIERS":    "34",  # Hérault
    "BORDEAUX":   "33",  # Gironde
    "BOURGES":    "18",  # Cher
    "BREST":      "29",  # Finistère
    "BRIANCON":   "05",  # Hautes-Alpes
    "CAEN":       "14",  # Calvados
    "CARCASS":    "11",  # Aude (Carcassonne)
    "CHAMBERY":   "73",  # Savoie
    "CHARLEV":    "08",  # Ardennes (Charleville-Mézières)
    "CHARTRES":   "28",  # Eure-et-Loir
    "CHERBOU":    "50",  # Manche (Cherbourg)
    "CLERMON":    "63",  # Puy-de-Dôme (Clermont-Ferrand)
    "COLMAR":     "68",  # Haut-Rhin
    "CREIL":      "60",  # Oise
    "DIJON":      "21",  # Côte-d'Or
    "DIGNE":      "04",  # Alpes-de-Haute-Provence
    "DUNKERQ":    "59",  # Nord (Dunkerque)
    "EMBRUN":     "05",  # Hautes-Alpes
    "GRENOBLE":   "38",  # Isère
    "LAON":       "02",  # Aisne
    "LAROCHE":    "89",  # Yonne (Laroche-Saint-Cydroine / Auxerre)
    "LAVAL":      "53",  # Mayenne
    "LEMANS":     "72",  # Sarthe (Le Mans)
    "LENS":       "62",  # Pas-de-Calais
    "LILE":       "59",  # Nord (Lille)
    "LIMOGES":    "87",  # Haute-Vienne
    "LORIENT":    "56",  # Morbihan
    "LYON":       "69",  # Rhône
    "MARSEILL":   "13",  # Bouches-du-Rhône
    "METZ":       "57",  # Moselle
    "MONTLUEL":   "01",  # Ain
    "MONTPELL":   "34",  # Hérault (Montpellier)
    "MULHOUSE":   "68",  # Haut-Rhin
    "NANCY":      "54",  # Meurthe-et-Moselle
    "NANTES":     "44",  # Loire-Atlantique
    "NICE":       "06",  # Alpes-Maritimes
    "NIMES":      "30",  # Gard
    "NIORT":      "79",  # Deux-Sèvres
    "ORLEANS":    "45",  # Loiret
    "PARIS":      "75",  # Paris
    "PAU":        "64",  # Pyrénées-Atlantiques
    "PERPIGNA":   "66",  # Pyrénées-Orientales (Perpignan)
    "POITIERS":   "86",  # Vienne
    "REIMS":      "51",  # Marne
    "RENNES":     "35",  # Ille-et-Vilaine
    "ROUEN":      "76",  # Seine-Maritime
    "SAINTBRI":   "22",  # Côtes-d'Armor (Saint-Brieuc) — corrigé depuis 71
    "SAINTETIE":  "42",  # Loire (Saint-Étienne)
    "SAINTMAL":   "35",  # Ille-et-Vilaine (Saint-Malo)
    "SAINTQUEN":  "80",  # Somme (Saint-Quentin)
    "SARREBOU":   "57",  # Moselle (Sarreboug)
    "STALBAN":    "81",  # Tarn (Saint-Alban)
    "STEFOY":     "33",  # Gironde (Sainte-Foy-la-Grande)
    "STRASBOU":   "67",  # Bas-Rhin (Strasbourg)
    "TOULON":     "83",  # Var
    "TOULOUSE":   "31",  # Haute-Garonne
    "TOULOUSM":   "31",  # Haute-Garonne (autre station Toulouse)
    "TOURS":      "37",  # Indre-et-Loire
    "TROYES":     "10",  # Aube
    "VALENCE":    "26",  # Drôme
    "VANNES":     "56",  # Morbihan
    "VERSAIL":    "78",  # Yvelines (Versailles)
    "VICHY":      "03",  # Allier
    "VIENNE":     "38",  # Isère (Vienne)
    "VILLENEU":   "47",  # Lot-et-Garonne (Villeneuve-sur-Lot)
    "ST-BRIEU":   "22",  # Côtes-d'Armor (Saint-Brieuc)
    "BLETTERA":   "07",  # Ardèche (Bletterans → 39 Jura ?)
    "MONTBRIS":   "42",  # Loire (Montbrison)
    "AUXERRE":    "89",  # Yonne
    "CHALONS":    "51",  # Marne (Châlons-en-Champagne)
    "CHARLEV2":   "08",  # Ardennes
    "BELFORT":    "90",  # Territoire de Belfort
    "MONTELIM":   "26",  # Drôme (Montélimar)
    "RODEZ":      "12",  # Aveyron
    "TARBES":     "65",  # Hautes-Pyrénées
    "BASTIA":     "2B",  # Haute-Corse
    "TOUQUET":    "62",  # Pas-de-Calais (Le Touquet)
    "BIARRIT":    "64",  # Pyrénées-Atlantiques (Biarritz)
    "CAHORS":     "46",  # Lot
    "PERIGUEU":   "24",  # Dordogne (Périgueux)
    "EVREUX":     "27",  # Eure
    "ABBEVIL":    "80",  # Somme (Abbeville)
    "ALENCON":    "61",  # Orne
    "CHAUMONT":   "52",  # Haute-Marne
    "COLOGNAC":   "30",  # Gard (Colognac)
    # A compléter : certaines villes du dataset n'ont pas de correspondance départementale (ex: Bletterans, Colognac, etc.) — à vérifier
}

# Pollen clés pour les pathologies respiratoires
POLLEN_CLES = {
    # TODO : liste complète ??
    
    "AMBROSIA":  "ambrosia",   # Ambroisie  → allergie sévère (août-sept)
    "ALNUS":     "alnus",      # Aulne/Aulnaie → allergie hiver-printemps
    "BETULA":    "betula",     # Bouleau    → allergie printemps
    "ARTEMISI":  "artemisia",  # Armoise    → allergie été
    "GRAMINE":   "graminees",  # Graminées  → allergie été (le + important)
    "GRAMINEES": "graminees",  # variante du nom
    "POACEAE":   "graminees",  # variante scientifique
    "CUPRESSU":  "cypres",     # Cyprès     → allergie hiver (Sud)
    "PLATANUS":  "platane",    # Platane    → allergie printemps (villes)
    "URTICA":    "urticacees", # Urticacées → allergie été
}

# Moisissures clés (allergènes fongiques, mêmes fichiers RNSA, colonnes séparées du pollen) 
MOISISSURE_CLES = {
     # TODO : liste complète ?? Difficile à comprendre...
     
    "ALTERNAR": "alternaria",    # Alternaria   → moisissure très allergisante, asthme sévère
    "CLADOSPO": "cladosporium",  # Cladosporium → moisissure extérieure la plus répandue
}


In [4]:

def build_dim_pollen() -> pd.DataFrame:
    """
    Construit la table dim_pollen à partir des fichiers RNSA BDD_daily.

    STRATÉGIE :
    ───────────
    1. Charger chaque fichier (ville × année)
    2. Rattacher la ville à un département via VILLE_TO_DEPT : liste à compléter ?? Y a-t-il une méthode plus pertinente ??
    3. Sélectionner uniquement les colonnes de pollens clés
    4. Agréger au mois → moyenne mensuelle par département
    5. Calculer des indicateurs synthétiques de risque pollinique

    VARIABLES PRODUITES :
    ────────────────────
    dept                → code département
    annee_mois          → période (ex: "2021-03")
    pollen_ambrosia_moy → concentration moyenne ambroisie (grains/m³)
    pollen_betula_moy   → concentration moyenne bouleau
    pollen_graminees_moy→ concentration moyenne graminées
    pollen_alnus_moy    → concentration moyenne aulne
    pollen_artemisia_moy→ concentration moyenne armoise
    pollen_global_max   → concentration max tous pollens confondus
    moisissure_alternaria_moy   → concentration moyenne Alternaria (grains/m³)
    moisissure_cladosporium_moy → concentration moyenne Cladosporium (grains/m³)

    CLÉ PRIMAIRE : dept × annee_mois
    """

    pollen_dir = RAW_DIR / "BDD_daily_2020_2025"
    if not pollen_dir.exists():
        print(f"⚠️  Dossier manquant : {pollen_dir}")
        return pd.DataFrame()

    dfs = []
    nb_ok = 0
    nb_err = 0

    # 2 formats possibles : .xls (anciennes années) ou .xlsx (années récentes)
    fichiers = sorted(pollen_dir.glob("*.xls")) + \
        sorted(pollen_dir.glob("*.xlsx"))
    print(f"Chargement de {len(fichiers)} fichiers RNSA...")

    for fpath in fichiers:
        # Extraire ville et année depuis le nom de fichier
        match = re.search(
            r'Particle_Extract_(.+?)_(\d{4})-\d{2}-\d{2}', fpath.name)
        if not match:
            continue

        ville = match.group(1)
        annee = int(match.group(2))

        # Rattacher au département
        dept = VILLE_TO_DEPT.get(ville)
        if dept is None:
            continue  # ville inconnue ou hors France (ex: ANDORRA)

        try:
            df = pd.read_excel(fpath)

            # Première colonne = date
            col_date = df.columns[0]
            df = df.rename(columns={col_date: "date"})
            df["date"] = pd.to_datetime(df["date"], errors="coerce")
            df = df.dropna(subset=["date"])

            # Filtrage temporel
            df = df[(df["date"].dt.year >= ANNEE_DEBUT) &
                    (df["date"].dt.year <= ANNEE_FIN)]

            if df.empty:
                continue

            df["annee_mois"] = df["date"].dt.to_period("M").astype(str)
            df["dept"] = dept

            # Identifier les colonnes de pollens clés disponibles
            cols_pollen = {}
            for col in df.columns:
                col_upper = col.upper() # même format des colonnes
                for nom_brut, nom_propre in POLLEN_CLES.items():
                    if nom_brut in col_upper:
                        df[col] = pd.to_numeric(df[col], errors="coerce")
                        cols_pollen[col] = f"pollen_{nom_propre}"
                        break

            # Identifier les colonnes de moisissures clés disponibles
            cols_moisissure = {}
            for col in df.columns:
                col_upper = col.upper() # même format des colonnes
                for nom_brut, nom_propre in MOISISSURE_CLES.items():
                    if nom_brut in col_upper:
                        df[col] = pd.to_numeric(df[col], errors="coerce")
                        cols_moisissure[col] = f"moisissure_{nom_propre}"
                        break

            if not cols_pollen and not cols_moisissure:
                continue

            # Renommer les colonnes de pollens et moisissures
            # (2 appels de suite : rename ne touche que les colonnes citées
            # dans le dictionnaire, les autres colonnes ne bougent pas)
            df = df.rename(columns=cols_pollen)
            df = df.rename(columns=cols_moisissure)

            # On liste les colonnes qu'on veut garder
            cols_utiles = ["dept", "annee_mois"]
            for nouveau_nom in cols_pollen.values():
                if nouveau_nom not in cols_utiles:
                    cols_utiles.append(nouveau_nom)
            for nouveau_nom in cols_moisissure.values():
                if nouveau_nom not in cols_utiles:
                    cols_utiles.append(nouveau_nom)

            # On ne garde que celles qui existent vraiment dans df
            colonnes_presentes = []
            for colonne in cols_utiles:
                if colonne in df.columns:
                    colonnes_presentes.append(colonne)
            df = df[colonnes_presentes]

            # Indicateur : concentration max tous pollens confondus
            pollen_cols_vals = [c for c in df.columns
                                if c.startswith("pollen_")]
            if pollen_cols_vals:
                df["pollen_global_max"] = df[pollen_cols_vals].max(axis=1)

            dfs.append(df)
            nb_ok += 1

        except Exception as e:
            nb_err += 1

    if not dfs:
        print("❌ Aucun fichier traité")
        return pd.DataFrame()

    print(f"✅ {nb_ok} fichiers chargés | {nb_err} erreurs")

    # Concaténer tous les fichiers
    df_all = pd.concat(dfs, ignore_index=True)
    print(f"Total lignes brutes : {len(df_all):,}")

    # Agrégation mensuelle par département
    # Plusieurs villes peuvent appartenir au même département
    # on fait la moyenne de toutes les stations du même dept × mois
    agg_dict = {}
    for col in df_all.columns:
        if col.startswith("pollen_") and col != "pollen_global_max":
            agg_dict[f"{col}_moy"] = (col, "mean")
    for col in df_all.columns:
        if col.startswith("moisissure_"):
            agg_dict[f"{col}_moy"] = (col, "mean")
    if "pollen_global_max" in df_all.columns:
        agg_dict["pollen_global_max"] = ("pollen_global_max", "max")

    # 1. on aggrège avec le format classique {colonne_source: fonction}
    colonnes_pour_agg = {}
    for nouveau_nom, (colonne_source, fonction) in agg_dict.items():
        colonnes_pour_agg[colonne_source] = fonction

    df_agg = (
        df_all.groupby(["dept", "annee_mois"])
        .agg(colonnes_pour_agg)
        .reset_index()
    )

    # 2. puis on renomme les colonnes obtenues avec les noms voulus
    noms_a_renommer = {}
    for nouveau_nom, (colonne_source, fonction) in agg_dict.items():
        noms_a_renommer[colonne_source] = nouveau_nom
    df_agg = df_agg.rename(columns=noms_a_renommer)

    # Arrondi
    for col in df_agg.select_dtypes(include="float").columns:
        df_agg[col] = df_agg[col].round(2)

    print(
        f"\n✅ dim_pollen : {df_agg.shape[0]:,} lignes × {df_agg.shape[1]} colonnes")
    print(
        f"Période      : {df_agg['annee_mois'].min()} → {df_agg['annee_mois'].max()}")
    print(f"Départements : {df_agg['dept'].nunique()} couverts")
    print(f"Colonnes     : {list(df_agg.columns)}")

    # Couverture par département
    depts_couverts = set(df_agg["dept"].unique())
    depts_manquants = set(DEPTS) - depts_couverts
    print(f"\n⚠️  {len(depts_manquants)} depts sans données pollen :")
    print(f"   {sorted(depts_manquants)}")
    print("   → Ces depts seront NaN dans df_model (normal)")

    return df_agg

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# EXÉCUTION
# ══════════════════════════════════════════════════════════════════════════════
print("Construction de dim_pollen...")
dim_pollen = build_dim_pollen()

if not dim_pollen.empty:
    valider_dim_table(dim_pollen, "dim_pollen")
    dim_pollen.to_parquet(TABLES_DIR / "dim_pollen.parquet", index=False)
    print(f"\n Sauvegardé → data/processed/dim_pollen.parquet")
    display(dim_pollen.head(10))

Construction de dim_pollen...
Chargement de 333 fichiers RNSA...


/opt/homebrew/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/opt/homebrew/lib/python3.9/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no d

✅ 203 fichiers chargés | 0 erreurs
Total lignes brutes : 73,981

✅ dim_pollen : 2,208 lignes × 12 colonnes
Période      : 2020-01 → 2024-12
Départements : 50 couverts
Colonnes     : ['dept', 'annee_mois', 'pollen_alnus_moy', 'pollen_ambrosia_moy', 'pollen_artemisia_moy', 'pollen_betula_moy', 'pollen_graminees_moy', 'pollen_platane_moy', 'pollen_urticacees_moy', 'moisissure_alternaria_moy', 'moisissure_cladosporium_moy', 'pollen_global_max']

⚠️  46 depts sans données pollen :
   ['02', '04', '05', '08', '09', '11', '12', '17', '19', '23', '27', '28', '2B', '32', '36', '39', '40', '41', '42', '43', '46', '48', '50', '53', '55', '58', '59', '60', '61', '62', '63', '64', '65', '70', '71', '77', '78', '82', '85', '88', '89', '90', '91', '93', '94', '95']
   → Ces depts seront NaN dans df_model (normal)
── Validation de dim_pollen ──
  ✅ Tous les codes dept sont valides (50 départements)
  ✅ Aucun doublon sur la clé ['dept', 'annee_mois']
  Taux de valeurs manquantes :
    pollen_alnus_moy 

,dept,annee_mois,pollen_alnus_moy,pollen_ambrosia_moy,pollen_artemisia_moy,pollen_betula_moy,pollen_graminees_moy,pollen_platane_moy,pollen_urticacees_moy,moisissure_alternaria_moy,moisissure_cladosporium_moy,pollen_global_max
0,01,2020-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01,2020-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,01,2020-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,01,2020-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,01,2020-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,01,2020-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,01,2020-07,0.0,1.33,1.13,0.0,3.30,0.0,15.68,NaN,NaN,45.51
7,01,2020-08,0.0,8.99,1.74,0.0,2.89,0.0,17.08,NaN,NaN,63.13
8,01,2020-09,0.0,37.19,0.63,0.0,2.22,0.0,5.04,NaN,NaN,110.82
9,01,2020-10,0.0,1.71,0.08,0.0,0.94,0.0,0.52,NaN,NaN,9.75
